In [ ]:
# DINOv3 ConvNeXt-L block for "Lost in the Museum"
#
# Why a ConvNeXt and not another ViT: the measured progression says architecture
# diversity is worth ~4x scale diversity.
#
#   DINOv3-H+ replacing ViT-g      +0.0067   better model, same family
#   + ViT-g back alongside it      +0.0403   DIFFERENT ARCHITECTURE
#   + DINOv3-L@512                 +0.0101   same family, different scale
#
# ConvNeXt-L is architecturally distinct from every block already in the concat
# (all ViTs) while inheriting DINOv3's training, so it should clear the
# ~0.78-0.79 standalone bar that a block must pass to earn its dimensions. A
# CLIP-family model would be MORE diverse but is semantic rather than
# instance-discriminative, and a sub-bar block costs real score -- ViT-L@392
# (0.775 alone) cost 0.0134.
#
# Attach: yihengwang66/dinov3-convnext-large-pretrain-lvd1689m   (no HF token)
import glob, os, time
import numpy as np, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None
SIZE  = 512          # ConvNeXt is fully convolutional -- any size works
HALF  = False        # fp32: fp16 returned 40,960,000 NaNs for DINOv3 ViT-L@512.
                     # ConvNeXt-L is only ~198M params, so fp32 is cheap here.
BATCH = 8
WORK  = Path('/kaggle/working')
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'CPU (attach a GPU)')

DATA = Path('/kaggle/input/competitions/lost-in-the-museum-f1/archive/kaggle_dataset/kaggle_dataset')
if not DATA.exists():
    hits = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
            if glob.glob(os.path.join(d, '*.png'))]
    assert hits, 'no PNG directory under /kaggle/input'
    DATA = Path(max(hits, key=lambda h: len(glob.glob(os.path.join(h, '*.png')))))
paths = sorted(DATA.glob('*.png'))
print(len(paths), 'images from', DATA)
assert len(paths) == 20000, f'expected 20000, got {len(paths)}'
CKPT, DONE = WORK / 'cnx_ckpt.npy', WORK / 'cnx_done.npy' 

In [ ]:
from transformers import AutoModel

dirs = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
        if os.path.exists(os.path.join(d, 'config.json'))]
print('attached model dirs:')
for d in dirs: print('   ', d)
pick = [d for d in dirs if 'convnext' in d.lower()] or dirs
assert pick, 'attach the DINOv3 ConvNeXt mirror'
model = AutoModel.from_pretrained(pick[0].rstrip('/')).eval().to(DEV)
if HALF and DEV == 'cuda': model = model.half()
print('loaded', pick[0])

# A ConvNeXt has NO cls token and NO register tokens -- the ViT code path would
# silently read the wrong thing. Its output is a feature MAP (B, C, H, W), so the
# CLS+GeM recipe becomes global-average + GeM over the spatial grid.
with torch.no_grad():
    probe = model(pixel_values=torch.zeros(1, 3, SIZE, SIZE, device=DEV,
                                           dtype=torch.half if (HALF and DEV=='cuda') else torch.float))
h = probe.last_hidden_state
print('output', tuple(h.shape))

# The HF wrapper can return EITHER layout, so detect rather than assume:
#   (B, C, H, W)  raw ConvNeXt feature map
#   (B, T, C)     unified sequence layout -- DINOv3 ConvNeXt returns this,
#                 e.g. (1, 257, 1536) = 1 global token + 16x16 spatial tokens
import math
if h.ndim == 4:
    SEQ, CH, NSP = False, h.shape[1], h.shape[2] * h.shape[3]
else:
    assert h.ndim == 3, f'unexpected output rank {h.ndim}: {tuple(h.shape)}'
    SEQ, CH = True, h.shape[2]
    NSP = math.isqrt(h.shape[1]) ** 2          # largest perfect square <= T
    assert NSP > 0 and h.shape[1] - NSP < 8, (
        f'cannot split {h.shape[1]} tokens into a square grid plus a few extras')
print(f'layout={"sequence" if SEQ else "map"}  channels {CH}  spatial {NSP}'
      f'  -> block width {2*CH}')

In [ ]:
tf = transforms.Compose([
    transforms.Resize((SIZE, SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __len__(self): return len(paths)
    def __getitem__(self, i):
        try: return tf(Image.open(paths[i]).convert('RGB')), i
        except Exception: return torch.zeros(3, SIZE, SIZE), i

@torch.no_grad()
def embed(x):
    h_ = model(pixel_values=x).last_hidden_state.float()
    if SEQ:
        # (B, T, C): token 0 is the global summary, the LAST NSP are spatial --
        # taking the last N is robust to however many extra tokens lead.
        glob = h_[:, 0]
        sp   = h_[:, -NSP:]                                # (B, NSP, C)
        gem  = sp.clamp(min=1e-6).pow(3.0).mean(1).pow(1/3.0)
    else:
        glob = h_.mean(dim=(2, 3))
        gem  = h_.clamp(min=1e-6).pow(3.0).mean(dim=(2, 3)).pow(1/3.0)
    return torch.cat([glob, gem], 1)

feats = np.load(CKPT) if CKPT.exists() else np.zeros((len(paths), 2*CH), np.float32)
done  = np.load(DONE) if DONE.exists() else np.zeros(len(paths), bool)
for pat in ('cnx_ckpt.npy', 'cnx_done.npy'):
    hit = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
    if hit and not CKPT.exists():
        a = np.load(hit[0])
        if pat.endswith('ckpt.npy') and a.shape == feats.shape: feats = a
        if pat.endswith('done.npy')  and a.shape == done.shape:  done = a
print(f'resuming with {done.sum()}/{len(paths)} embedded', flush=True)

todo = np.flatnonzero(~done)
dl = DataLoader(DS(), batch_size=BATCH, num_workers=2, sampler=todo.tolist())
t0 = time.time(); n = 0
for x, idx in dl:
    x = x.to(DEV)
    if HALF and DEV == 'cuda': x = x.half()
    v = embed(x).cpu().numpy()
    if n == 0:
        assert np.isfinite(v).all(), ('first batch non-finite -- set HALF=False '
                                      '(fp16 overflow produced all-NaN features before)')
    feats[idx.numpy()] = v; done[idx.numpy()] = True
    n += len(idx)
    if n % (BATCH*100) == 0:
        r = n/(time.time()-t0)
        print(f'  {done.sum()}/{len(paths)}  {r:.1f} img/s  ETA {(len(todo)-n)/r/60:.0f} min', flush=True)
        np.save(CKPT, feats); np.save(DONE, done)

np.save(CKPT, feats); np.save(DONE, done)
assert np.isfinite(feats).all(), 'non-finite features -- do not use this file'
np.save(WORK / 'features_cnx512.npy', feats)
print(f'wrote features_cnx512.npy {feats.shape} in {(time.time()-t0)/60:.1f} min')
print('finite:', np.isfinite(feats).all(), '| zero rows:',
      int((np.linalg.norm(feats, axis=1) == 0).sum()))